# Amazon S3 Tables PoC — Multi-Engine Access & Batch Ingestion

This notebook validates that PyIceberg can read and write the same S3 Tables that Athena uses — confirming true multi-engine interoperability via the S3 Tables REST endpoint.

**What you'll validate:**
- PyIceberg connects to S3 Tables using SigV4 authentication
- Data written by Athena is readable by PyIceberg (and vice versa)
- Batch ingestion of 50K records via PyIceberg
- Cross-engine query verification via Athena

**Prerequisites:**
- Deploy the CloudFormation stack (see README Step 2)
- Complete Phase 1.1–1.3 (catalog, namespace, and CRUD via Athena)
- AWS credentials configured locally
- `pip install "pyiceberg[s3,pyarrow]" boto3 pyarrow pandas`

## Setup

Install dependencies (skip if already installed):

In [ ]:
!pip install -q "pyiceberg[s3,pyarrow]" boto3 pyarrow pandas

### Configuration

Auto-discover resource names from CloudFormation outputs. Update `AWS_REGION` and `STACK_NAME` if you used different values during deployment:

In [ ]:
import boto3
import json

# Configuration — update these to match your deployment
AWS_REGION = "us-east-1"
STACK_NAME = "s3-tables-poc"

# Auto-discover from CloudFormation outputs
cfn = boto3.client("cloudformation", region_name=AWS_REGION)
outputs = {o["OutputKey"]: o["OutputValue"] 
           for o in cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]["Outputs"]}

TABLE_BUCKET_ARN = outputs["TableBucketARN"]
TABLE_BUCKET_NAME = outputs["TableBucketName"]
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

print(f"Region:       {AWS_REGION}")
print(f"Table Bucket: {TABLE_BUCKET_ARN}")
print(f"Account:      {ACCOUNT_ID}")

## 1. Connect to S3 Tables via PyIceberg REST Catalog

PyIceberg connects to S3 Tables using the REST catalog endpoint with SigV4 authentication. This is the same endpoint Spark uses — no Glue catalog dependency for direct table access.

In [ ]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    "s3tables",
    type="rest",
    warehouse=TABLE_BUCKET_ARN,
    uri=f"https://s3tables.{AWS_REGION}.amazonaws.com/iceberg",
    **{
        "rest.sigv4-enabled": "true",
        "rest.signing-name": "s3tables",
        "rest.signing-region": AWS_REGION,
    }
)

# List namespaces — should show 'poc_data' created in Phase 1.2
print("Namespaces:", catalog.list_namespaces())

## 2. Cross-Engine Read — Read Athena-Created Table

The `customers` table was created and populated by Athena in Phase 1.3. Reading it here confirms that PyIceberg can access data written by a different engine — both go through the same Iceberg metadata.

In [ ]:
# List tables in poc_data namespace
print("Tables:", catalog.list_tables("poc_data"))

In [ ]:
# Read the 'customers' table created by Athena
customers = catalog.load_table("poc_data.customers")
df = customers.scan().to_pandas()
print(f"Rows from Athena-created table: {len(df)}")
df.sort_values("id")

## 3. Cross-Engine Write — Write from PyIceberg

Write new rows from PyIceberg into the same `customers` table. After this cell, you can verify in Athena that the new rows appear — confirming bidirectional multi-engine access.

> **Note**: The `id` column must use `pa.int32()` to match the Athena `INT` type. PyArrow defaults to `int64` which causes a schema mismatch.

In [ ]:
import pyarrow as pa
from datetime import datetime

# Write new rows from PyIceberg
new_rows = pa.table({
    "id": pa.array([10, 11], type=pa.int32()),
    "name": ["PyIceberg User 1", "PyIceberg User 2"],
    "email": ["pyiceberg1@example.com", "pyiceberg2@example.com"],
    "created_at": [datetime.now(), datetime.now()],
})

customers.append(new_rows)
print("Appended 2 rows from PyIceberg")

# Verify
df = customers.scan().to_pandas()
print(f"Total rows now: {len(df)}")
df.sort_values("id")

## 4. Create Events Table with Partitioning

Create a partitioned `events` table for batch ingestion testing. Day-level partitioning on `event_time` is a common pattern for time-series data — it enables partition pruning on time-range queries.

> Fields use `required=False` (nullable) to match PyArrow's default behavior and avoid schema compatibility errors on append.

In [ ]:
from pyiceberg.schema import Schema
from pyiceberg.types import (
    StringType, IntegerType, DoubleType, TimestampType, NestedField
)
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import DayTransform

# Define schema
events_schema = Schema(
    NestedField(1, "event_id", StringType(), required=False),
    NestedField(2, "event_type", StringType(), required=False),
    NestedField(3, "user_id", IntegerType(), required=False),
    NestedField(4, "amount", DoubleType(), required=False),
    NestedField(5, "event_time", TimestampType(), required=False),
    NestedField(6, "region", StringType(), required=False),
)

# Partition by day(event_time)
partition_spec = PartitionSpec(
    PartitionField(source_id=5, field_id=1000, transform=DayTransform(), name="event_day")
)

# Create table
try:
    events_table = catalog.create_table(
        "poc_data.events",
        schema=events_schema,
        partition_spec=partition_spec,
    )
    print("Created events table")
except Exception as e:
    if "already exists" in str(e).lower():
        events_table = catalog.load_table("poc_data.events")
        print("Events table already exists, loaded it")
    else:
        raise

## 5. Batch Ingestion — Load 50K Records

Generate synthetic event data and load it in batches of 10K. This validates that S3 Tables handles concurrent writes and that the automatic compaction will later merge the resulting small files.

> `user_id` uses `pa.int32()` to match the Iceberg `IntegerType` in the schema.

In [ ]:
import random
import uuid
from datetime import datetime, timedelta

EVENT_TYPES = ["purchase", "click", "view", "refund", "signup"]
REGIONS = ["us-east-1", "eu-west-1", "ap-southeast-1", "us-west-2"]
NUM_RECORDS = 50_000
BATCH_SIZE = 10_000

print(f"Generating and loading {NUM_RECORDS:,} records in batches of {BATCH_SIZE:,}...")

for batch_start in range(0, NUM_RECORDS, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, NUM_RECORDS)
    n = batch_end - batch_start
    
    batch = pa.table({
        "event_id": [str(uuid.uuid4()) for _ in range(n)],
        "event_type": [random.choice(EVENT_TYPES) for _ in range(n)],
        "user_id": pa.array([random.randint(1, 1000) for _ in range(n)], type=pa.int32()),
        "amount": [round(random.uniform(1.0, 500.0), 2) for _ in range(n)],
        "event_time": [
            datetime.now() - timedelta(days=random.randint(0, 6), 
                                       hours=random.randint(0, 23),
                                       minutes=random.randint(0, 59))
            for _ in range(n)
        ],
        "region": [random.choice(REGIONS) for _ in range(n)],
    })
    
    events_table.append(batch)
    print(f"  Loaded batch {batch_start+1:,}–{batch_end:,}")

print(f"\nDone. Total records loaded: {NUM_RECORDS:,}")

## 6. Verify Batch Load

Read back the data and check aggregates to confirm all records were written correctly:

In [ ]:
# Verify the load
events_table = catalog.load_table("poc_data.events")
df = events_table.scan().to_pandas()
print(f"Total rows in events table: {len(df):,}")
print(f"\nBy event_type:")
print(df.groupby("event_type").agg({"event_id": "count", "amount": "sum"}).rename(columns={"event_id": "count"}))
print(f"\nBy region:")
print(df.groupby("region")["event_id"].count())

## 7. Cross-Engine Verification — Query from Athena

Run the same aggregation query via Athena to confirm it can read the PyIceberg-written data. This proves true multi-engine interoperability — data written by one engine is immediately queryable by another.

In [ ]:
import time

athena = boto3.client("athena", region_name=AWS_REGION)

query = "SELECT event_type, count(*) as cnt, round(sum(amount),2) as total_amount FROM events GROUP BY event_type ORDER BY cnt DESC"

response = athena.start_query_execution(
    QueryString=query,
    QueryExecutionContext={"Catalog": f"s3tablescatalog/{TABLE_BUCKET_NAME}", "Database": "poc_data"},
    WorkGroup=outputs["AthenaWorkgroupName"]
)

qid = response["QueryExecutionId"]
while True:
    status = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
    if status in ("SUCCEEDED", "FAILED", "CANCELLED"):
        break
    time.sleep(1)

if status == "SUCCEEDED":
    results = athena.get_query_results(QueryExecutionId=qid)
    print("Cross-engine verification (Athena reading PyIceberg-written data):")
    for row in results["ResultSet"]["Rows"]:
        print("  ", [col.get("VarCharValue", "") for col in row["Data"]])
else:
    reason = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"].get("StateChangeReason", "")
    print(f"Query {status}: {reason}")

## 8. Table Metadata & Maintenance Status

Inspect the table's Iceberg metadata (schema, partitioning, snapshot history) and check whether S3 Tables has run any automatic maintenance jobs (compaction, snapshot management).

> Maintenance jobs run asynchronously. With 50K records you may not see compaction immediately — check back after an hour. A status of `Not_Yet_Run` is expected on a fresh deployment.

In [ ]:
# Inspect table metadata
events_table = catalog.load_table("poc_data.events")

print(f"Schema: {events_table.schema()}")
print(f"Partition spec: {events_table.spec()}")
print(f"Current snapshot: {events_table.current_snapshot()}")
print(f"\nSnapshot history:")
for snap in events_table.history():
    print(f"  {snap}")

In [ ]:
# Check maintenance status via S3 Tables API
s3tables = boto3.client("s3tables", region_name=AWS_REGION)

maintenance = s3tables.get_table_maintenance_configuration(
    tableBucketARN=TABLE_BUCKET_ARN,
    namespace="poc_data",
    name="events"
)
print("Maintenance configuration:")
print(json.dumps(maintenance["configuration"], indent=2, default=str))

job_status = s3tables.get_table_maintenance_job_status(
    tableBucketARN=TABLE_BUCKET_ARN,
    namespace="poc_data",
    name="events"
)
print("\nMaintenance job status:")
print(json.dumps(job_status["status"], indent=2, default=str))

---

**Done!** You've validated:
- ✅ PyIceberg connects to S3 Tables via REST + SigV4
- ✅ Cross-engine reads (Athena → PyIceberg)
- ✅ Cross-engine writes (PyIceberg → Athena)
- ✅ Batch ingestion (50K records)
- ✅ Table metadata and maintenance observability

**Next steps**: Return to the README for Phase 2 (Firehose streaming), Phase 3 (observability), and Phase 4 (administration).